In [1]:
import pandas as pd
from pathlib import Path

# Load manually pasted FBRef schedules
csv_path = Path('/Users/admin/dev/algobetting/infra/data/json/fbref/fixtures_manual.csv')

df = pd.read_csv(csv_path, parse_dates=['date'])
df['date'] = pd.to_datetime(df['date']).dt.tz_localize(None)  # strip tz for simplicity

print(f'Total rows: {len(df)}')
print(f'\nBy competition:')
print(df['league_name'].value_counts().to_string())
print(f'\nBy season:')
print(df['season'].value_counts().sort_index().to_string())


Total rows: 8626

By competition:
league_name
Premier League       3359
Europa League        1592
FA Cup               1184
Champions League     1118
Conference League     714
EFL Cup               659

By season:
season
2017-2018     946
2018-2019     927
2019-2020     799
2020-2021     892
2021-2022     977
2022-2023     995
2023-2024     993
2024-2025    1088
2025-2026    1009


In [2]:
# Build a team-centric view: one row per team per fixture
# Synthetic fixture key from date + home + away (no fixture_id in manual CSV)

df['fixture_key'] = df['date'].astype(str) + '|' + df['home_team'] + '|' + df['away_team']

home = df[['fixture_key', 'date', 'season', 'league_name', 'home_team', 'away_team',
           'home_goals', 'away_goals']].copy()
home = home.rename(columns={'home_team': 'team', 'away_team': 'opponent',
                             'home_goals': 'gf', 'away_goals': 'ga'})
home['is_home'] = True

away = df[['fixture_key', 'date', 'season', 'league_name', 'away_team', 'home_team',
           'away_goals', 'home_goals']].copy()
away = away.rename(columns={'away_team': 'team', 'home_team': 'opponent',
                             'away_goals': 'gf', 'home_goals': 'ga'})
away['is_home'] = False

team_df = pd.concat([home, away], ignore_index=True)
team_df['date'] = pd.to_datetime(team_df['date'])
team_df = team_df.sort_values(['team', 'date']).reset_index(drop=True)

print(f'Team-centric rows: {len(team_df)}')
team_df.head()


Team-centric rows: 17252


,fixture_key,date,season,league_name,team,opponent,gf,ga,is_home
0,2020-10-22|PAOK|AC Omonia,2020-10-22,2020-2021,Europa League,AC Omonia,PAOK,1,1,False
1,2020-10-29|AC Omonia|PSV,2020-10-29,2020-2021,Europa League,AC Omonia,PSV,1,2,True
2,2020-11-05|AC Omonia|Granada,2020-11-05,2020-2021,Europa League,AC Omonia,Granada,0,2,True
3,2020-11-26|Granada|AC Omonia,2020-11-26,2020-2021,Europa League,AC Omonia,Granada,1,2,False
4,2020-12-03|AC Omonia|PAOK,2020-12-03,2020-2021,Europa League,AC Omonia,PAOK,2,1,True


In [3]:
# Drop team-seasons with fewer than 38 Premier League fixtures

pl_counts = (
    team_df[team_df['league_name'] == 'Premier League']
    .groupby(['team', 'season'])
    .size()
    .reset_index(name='pl_fixtures')
)

valid = pl_counts[pl_counts['pl_fixtures'] >= 38][['team', 'season']]
print(f'Valid team-seasons (>=38 PL fixtures): {len(valid)}')

dropped = pl_counts[pl_counts['pl_fixtures'] < 38]
if not dropped.empty:
    print('\nDropped team-seasons:')
    print(dropped.to_string(index=False))

team_df = team_df.merge(valid, on=['team', 'season'], how='inner')
print(f'\nTeam-centric rows after filter: {len(team_df)}')


Valid team-seasons (>=38 PL fixtures): 160

Dropped team-seasons:
             team    season  pl_fixtures
          Arsenal 2025-2026           32
      Aston Villa 2025-2026           32
      Bournemouth 2025-2026           32
        Brentford 2025-2026           32
         Brighton 2025-2026           32
          Burnley 2025-2026           32
          Chelsea 2025-2026           32
   Crystal Palace 2025-2026           31
          Everton 2025-2026           32
           Fulham 2025-2026           32
     Leeds United 2025-2026           32
        Liverpool 2025-2026           32
  Manchester City 2025-2026           31
   Manchester Utd 2025-2026           32
 Newcastle United 2025-2026           32
Nottingham Forest 2025-2026           32
       Sunderland 2025-2026           32
Tottenham Hotspur 2025-2026           32
  West Ham United 2025-2026           32
           Wolves 2025-2026           32

Team-centric rows after filter: 7460


In [4]:
import numpy as np
import datetime

# ---------------------------------------------------------------------------
# Calendar-based 'free midweek' detection
#
# For each PL game on Fri/Sat/Sun/Mon:
#   - Look at the Tue/Wed/Thu window in the 7 days before the game
#   - If the team played ANY game (any competition) on one of those days -> busy
#   - If no game Tue-Thu AND days_rest <= 10 -> free_midweek = True
#   - If days_rest > 10 -> exclude (long break)
#   - Tue/Wed/Thu PL games -> not classified (return NaN)
# ---------------------------------------------------------------------------

WEEKEND_DAYS = {4, 5, 6, 0}   # Fri=4, Sat=5, Sun=6, Mon=0
MIDWEEK_DAYS = {1, 2, 3}       # Tue=1, Wed=2, Thu=3

# Calculate days_rest per team (across all competitions)
team_df = team_df.sort_values(['team', 'date']).reset_index(drop=True)
team_df['days_rest'] = (
    team_df.groupby('team')['date']
    .diff()
    .dt.total_seconds()
    .div(86400)
    .round(1)
)

# Build lookup set: (team, date) for all games in the dataset
all_game_dates = set(
    zip(team_df['team'], team_df['date'].dt.date)
)

def classify_midweek(team, game_date, league, days_rest):
    if league != 'Premier League':
        return float('nan')
    dow = game_date.weekday()  # Mon=0 ... Sun=6
    if dow not in WEEKEND_DAYS:
        return float('nan')   # midweek PL fixture
    if days_rest is None or (days_rest != days_rest) or days_rest > 10:
        return float('nan')   # long break or season opener
    # Check 7 days before for any Tue/Wed/Thu game
    gd = game_date.date()
    for delta in range(1, 8):
        check = gd - datetime.timedelta(days=delta)
        if check.weekday() in MIDWEEK_DAYS:
            if (team, check) in all_game_dates:
                return False  # had a midweek game
    return True  # free midweek

team_df['free_midweek'] = [
    classify_midweek(r.team, r.date, r.league_name, r.days_rest)
    for r in team_df.itertuples()
]

pl_classifiable = team_df[team_df['free_midweek'].notna()]
print(f'Classifiable PL weekend games: {len(pl_classifiable)}')
print(f'\nfree_midweek breakdown:')
print(pl_classifiable['free_midweek'].value_counts())
print(f'\nSample rows:')
pl_classifiable[['team', 'date', 'league_name', 'days_rest', 'free_midweek']].head(10)


Classifiable PL weekend games: 4208

free_midweek breakdown:
free_midweek
True     2475
False    1733
Name: count, dtype: int64

Sample rows:


,team,date,league_name,days_rest,free_midweek
1,Arsenal,2017-08-19,Premier League,8.0,True
2,Arsenal,2017-08-27,Premier League,8.0,True
5,Arsenal,2017-09-17,Premier League,3.0,False
7,Arsenal,2017-09-25,Premier League,5.0,False
9,Arsenal,2017-10-01,Premier League,3.0,False
12,Arsenal,2017-10-22,Premier League,3.0,False
14,Arsenal,2017-10-28,Premier League,4.0,False
16,Arsenal,2017-11-05,Premier League,3.0,False
19,Arsenal,2017-11-26,Premier League,3.0,False
21,Arsenal,2017-12-02,Premier League,3.0,False


In [5]:
# Model prep: classifiable PL weekend games only
# Target: points (3 win, 1 draw, 0 loss)

pl = team_df[team_df['free_midweek'].notna()].copy()
pl['result'] = (pl['gf'] > pl['ga']).astype(int) * 3 + (pl['gf'] == pl['ga']).astype(int)

# Self-join on fixture_key to get opponent's free_midweek + days_rest
opp_cols = pl[['fixture_key', 'team', 'days_rest', 'free_midweek']].copy()
opp_cols.columns = ['fixture_key', 'opponent', 'opp_days_rest', 'opp_free_midweek']
pl = pl.merge(opp_cols, on=['fixture_key', 'opponent'], how='left')

pl['rest_advantage'] = pl['days_rest'] - pl['opp_days_rest']
# midweek_adv: +1 if only we had free midweek, -1 if only opponent did, 0 otherwise
pl['midweek_adv'] = pl['free_midweek'].astype(float) - pl['opp_free_midweek'].astype(float)

pl_model = pl.dropna(subset=['opp_days_rest', 'opp_free_midweek']).copy()
pl_model['is_home'] = pl_model['is_home'].astype(int)
pl_model['free_midweek'] = pl_model['free_midweek'].astype(int)  # bool -> 0/1 for statsmodels

print(f'Model rows: {len(pl_model)}')
print(f'\nPoints distribution:\n{pl_model["result"].value_counts().sort_index()}')
print(f'\nfree_midweek distribution:\n{pl_model["free_midweek"].value_counts()}')
print(f'\nmidweek_adv distribution:\n{pl_model["midweek_adv"].value_counts().sort_index()}')
print(f'\nMean points by free_midweek:')
print(pl_model.groupby('free_midweek')['result'].agg(['mean','count']).to_string())
pl_model[['team', 'date', 'days_rest', 'opp_days_rest', 'free_midweek',
          'opp_free_midweek', 'rest_advantage', 'midweek_adv', 'result']].head(10)


Model rows: 4128

Points distribution:
result
0    1586
1     956
3    1586
Name: count, dtype: int64

free_midweek distribution:
free_midweek
1    2435
0    1693
Name: count, dtype: int64

midweek_adv distribution:
midweek_adv
-1.0     681
 0.0    2766
 1.0     681
Name: count, dtype: int64

Mean points by free_midweek:
                  mean  count
free_midweek                 
0             1.506202   1693
1             1.299384   2435


,team,date,days_rest,opp_days_rest,free_midweek,opp_free_midweek,rest_advantage,midweek_adv,result
0,Arsenal,2017-08-19,8.0,7.0,1,True,1.0,0.0,0
1,Arsenal,2017-08-27,8.0,8.0,1,True,0.0,0.0,0
2,Arsenal,2017-09-17,3.0,5.0,0,False,-2.0,0.0,1
3,Arsenal,2017-09-25,5.0,5.0,0,False,0.0,0.0,3
4,Arsenal,2017-10-01,3.0,7.0,0,True,-4.0,-1.0,3
5,Arsenal,2017-10-22,3.0,3.0,0,False,0.0,0.0,3
6,Arsenal,2017-10-28,4.0,4.0,0,False,0.0,0.0,3
7,Arsenal,2017-11-05,3.0,4.0,0,False,-1.0,0.0,0
8,Arsenal,2017-11-26,3.0,8.0,0,True,-5.0,-1.0,3
9,Arsenal,2017-12-02,3.0,4.0,0,False,-1.0,0.0,0


In [6]:
import statsmodels.formula.api as smf

# M1: free_midweek binary, no fixed effects
m1 = smf.ols(
    'result ~ free_midweek + rest_advantage + is_home',
    data=pl_model
).fit()

# M2: + team FE
m2 = smf.ols(
    'result ~ free_midweek + rest_advantage + is_home + C(team)',
    data=pl_model
).fit()

# M3: midweek_adv + team FE
m3 = smf.ols(
    'result ~ midweek_adv + rest_advantage + is_home + C(team)',
    data=pl_model
).fit()

# M4: + team-season FE (interacted)
pl_model['team_season'] = pl_model['team'] + '_' + pl_model['season'].astype(str)
m4 = smf.ols(
    'result ~ free_midweek + rest_advantage + is_home + C(team_season)',
    data=pl_model
).fit()

# M5: team + season + opponent as separate FEs (~50 params vs ~320)
m5 = smf.ols(
    'result ~ free_midweek + rest_advantage + is_home + C(team) + C(season) + C(opponent)',
    data=pl_model
).fit()

def print_model(label, model, focus):
    print(f'=== {label} ===')
    for v in focus:
        if v in model.params.index:
            p = model.pvalues[v]
            sig = '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.1 else ''))
            print(f'  {v:<30} coef={model.params[v]:+.3f}  p={p:.3f}  {sig}')
    print(f'  R2 = {model.rsquared:.3f}  N = {int(model.nobs)}  params = {len(model.params)}')
    print()

focus = ['free_midweek', 'rest_advantage', 'is_home']
print_model('M1: no FE',                               m1, focus)
print_model('M2: + team FE',                           m2, focus)
print_model('M3: midweek_adv + team FE',               m3, ['midweek_adv', 'rest_advantage', 'is_home'])
print_model('M4: + team-season FE (interacted)',        m4, focus)
print_model('M5: + team + season + opponent FE (additive)', m5, focus)

=== M1: no FE ===
  free_midweek                   coef=-0.030  p=0.534  
  rest_advantage                 coef=-0.072  p=0.000  ***
  is_home                        coef=+0.337  p=0.000  ***
  R2 = 0.033  N = 4128  params = 4

=== M2: + team FE ===
  free_midweek                   coef=+0.119  p=0.012  **
  rest_advantage                 coef=-0.049  p=0.000  ***
  is_home                        coef=+0.341  p=0.000  ***
  R2 = 0.106  N = 4128  params = 34

=== M3: midweek_adv + team FE ===
  midweek_adv                    coef=-0.106  p=0.148  
  rest_advantage                 coef=-0.014  p=0.472  
  is_home                        coef=+0.337  p=0.000  ***
  R2 = 0.105  N = 4128  params = 34

=== M4: + team-season FE (interacted) ===
  free_midweek                   coef=+0.126  p=0.009  ***
  rest_advantage                 coef=-0.051  p=0.000  ***
  is_home                        coef=+0.335  p=0.000  ***
  R2 = 0.151  N = 4128  params = 163

=== M5: + team + season + opponent FE 

In [7]:
# --- Asymmetric matchup analysis ---
# Filter to games where exactly one team had a free midweek and the other didn't.
# In this subset: free_midweek=1 = "we were the rested side", free_midweek=0 = "we were the busy side"
# This is the cleanest direct test of the fatigue hypothesis.

asym = pl_model[pl_model['midweek_adv'] != 0].copy()
print(f"Asymmetric matchups (one team rested, one busy): {len(asym)} team-game rows")
print(f"  Rested side (free_midweek=1): {(asym['free_midweek']==1).sum()}")
print(f"  Busy side   (free_midweek=0): {(asym['free_midweek']==0).sum()}")

# Mean points and win rate by side
print("\n=== Raw means (no controls) ===")
summary = asym.groupby('free_midweek')['result'].agg(['mean','count'])
summary.index = ['Busy midweek', 'Free midweek']
summary.columns = ['mean_pts', 'n']
summary['win_rate'] = asym.groupby('free_midweek').apply(lambda x: (x['result']==3).mean()).values
print(summary.to_string())

# Regression on asymmetric subset with team-season FE
import statsmodels.formula.api as smf

asym['team_season'] = asym['team'] + '_' + asym['season'].astype(str)

ma = smf.ols(
    'result ~ free_midweek + is_home + C(team_season)',
    data=asym
).fit()

print("\n=== OLS on asymmetric matchups + team-season FE ===")
for v in ['free_midweek', 'is_home']:
    p = ma.pvalues[v]
    sig = '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.1 else ''))
    print(f"  {v:<30} coef={ma.params[v]:+.3f}  p={p:.3f}  {sig}")
print(f"  R2 = {ma.rsquared:.3f}  N = {int(ma.nobs)}")

Asymmetric matchups (one team rested, one busy): 1362 team-game rows
  Rested side (free_midweek=1): 681
  Busy side   (free_midweek=0): 681

=== Raw means (no controls) ===
              mean_pts    n  win_rate
Busy midweek  1.688693  681  0.484581
Free midweek  1.076358  681  0.280470

=== OLS on asymmetric matchups + team-season FE ===
  free_midweek                   coef=-0.239  p=0.004  ***
  is_home                        coef=+0.307  p=0.000  ***
  R2 = 0.259  N = 1362


In [8]:
# ---------------------------------------------------------------------------
# Load odds and compute expected points
# ---------------------------------------------------------------------------
# Requires infra/data/json/odds/odds_flat.csv
# Populate it by running infra/data/collectors/odds/parse_pasted_odds.ipynb

from pathlib import Path

ODDS_CSV = Path('/Users/admin/dev/algobetting/infra/data/json/odds/odds_flat.csv')

if not ODDS_CSV.exists():
    raise FileNotFoundError(
        f"No odds file at {ODDS_CSV}\n"
        "Run infra/data/collectors/odds/parse_pasted_odds.ipynb first."
    )

odds = pd.read_csv(ODDS_CSV, parse_dates=['date'])
odds['date'] = pd.to_datetime(odds['date']).dt.tz_localize(None)
print(f"Odds rows: {len(odds)}")
print(odds['season'].value_counts().sort_index().to_string())

# ---------------------------------------------------------------------------
# Team name mapping: football-data.co.uk -> FBRef names
# football-data uses short names; FBRef uses full names.
# Extend this dict as you add more seasons.
# ---------------------------------------------------------------------------
# Note: odds data starts from the 2018-19 season (Aug 2018);
# FBRef 2017-18 fixtures (Stoke, Swansea etc.) will have no odds match.
ODDS_TO_FBREF = {
    "Man City":        "Manchester City",
    "Man United":      "Manchester Utd",
    "Newcastle":       "Newcastle United",
    "Tottenham":       "Tottenham Hotspur",
    "West Ham":        "West Ham United",
    "Nott'm Forest":   "Nottingham Forest",
    "Sheffield United":"Sheffield United",   # same — keeps explicit
    "Cardiff":         "Cardiff City",
    "Huddersfield":    "Huddersfield Town",
    "West Brom":       "West Brom",
    "Wolves":          "Wolves",
    "Brighton":        "Brighton",
    "Leicester":       "Leicester City",
    "Leeds":           "Leeds United",
    "Norwich":         "Norwich City",
    "Brentford":       "Brentford",
    "Burnley":         "Burnley",
    "Watford":         "Watford",
    "Luton":           "Luton Town",
    "Ipswich":         "Ipswich Town",
    "Bournemouth":     "Bournemouth",
    "Southampton":     "Southampton",
    "Crystal Palace":  "Crystal Palace",
    "Everton":         "Everton",
    "Fulham":          "Fulham",
    "Chelsea":         "Chelsea",
    "Arsenal":         "Arsenal",
    "Liverpool":       "Liverpool",
    "Aston Villa":     "Aston Villa",
    "Sunderland":      "Sunderland",
}

odds['home_team_fbref'] = odds['home_team'].map(ODDS_TO_FBREF).fillna(odds['home_team'])
odds['away_team_fbref'] = odds['away_team'].map(ODDS_TO_FBREF).fillna(odds['away_team'])

# Build a fixture-level odds lookup keyed by (date.date, home_team_fbref, away_team_fbref)
odds['date_only'] = odds['date'].dt.date
odds_lookup = odds.set_index(['date_only', 'home_team_fbref', 'away_team_fbref'])[
    ['xpts_home', 'xpts_away', 'p_home', 'p_draw', 'p_away']
]

# ---------------------------------------------------------------------------
# Join to pl_model (already has fixture_key = 'date|home|away')
# ---------------------------------------------------------------------------
pl_model['date_only'] = pl_model['date'].dt.date

# Recover home/away from fixture_key (format: date|home_team|away_team)
pl_model[['fk_date', 'fk_home', 'fk_away']] = pl_model['fixture_key'].str.split('|', expand=True)

def lookup_xpts(row):
    key = (row['date_only'], row['fk_home'], row['fk_away'])
    if key in odds_lookup.index:
        o = odds_lookup.loc[key]
        # Return xpts for this team (home or away)
        return o['xpts_home'] if row['is_home'] else o['xpts_away']
    return float('nan')

pl_model['xpts'] = pl_model.apply(lookup_xpts, axis=1)

n_matched = pl_model['xpts'].notna().sum()
n_total   = len(pl_model)
print(f"\nMatched odds: {n_matched}/{n_total} ({100*n_matched/n_total:.1f}%)")

# Unmatched fixtures (useful for diagnosing name mapping gaps)
unmatched = pl_model[pl_model['xpts'].isna()][['date', 'team', 'opponent', 'fk_home', 'fk_away', 'fixture_key']].drop_duplicates('fixture_key')
if not unmatched.empty:
    print(f"\nUnmatched sample (check ODDS_TO_FBREF mapping):")
    print(unmatched.head(10).to_string(index=False))


Odds rows: 3359
season
2017-2018    380
2018-2019    380
2019-2020    380
2020-2021    380
2021-2022    380
2022-2023    380
2023-2024    380
2024-2025    380
2025-2026    319

Matched odds: 4022/4128 (97.4%)

Unmatched sample (check ODDS_TO_FBREF mapping):
      date        team     opponent      fk_home      fk_away                         fixture_key
2017-08-19     Arsenal   Stoke City   Stoke City      Arsenal       2017-08-19|Stoke City|Arsenal
2017-10-28     Arsenal Swansea City      Arsenal Swansea City     2017-10-28|Arsenal|Swansea City
2017-10-21 Bournemouth   Stoke City   Stoke City  Bournemouth   2017-10-21|Stoke City|Bournemouth
2017-11-25 Bournemouth Swansea City Swansea City  Bournemouth 2017-11-25|Swansea City|Bournemouth
2018-02-03 Bournemouth   Stoke City  Bournemouth   Stoke City   2018-02-03|Bournemouth|Stoke City
2018-05-05 Bournemouth Swansea City  Bournemouth Swansea City 2018-05-05|Bournemouth|Swansea City
2017-11-04    Brighton Swansea City Swansea City     Bri

In [9]:
# ---------------------------------------------------------------------------
# Points vs expectation model
# ---------------------------------------------------------------------------
# Target: pts_vs_xpts = actual_pts - expected_pts_from_odds
# The odds already encode team quality + opposition strength, so no
# team or opponent fixed effects are needed.
# Note: FBRef 2017-18 fixtures have no odds coverage (odds start 2018-19).
# ---------------------------------------------------------------------------

pl_odds = pl_model.dropna(subset=['xpts']).copy()
pl_odds['pts_vs_xpts'] = pl_odds['result'] - pl_odds['xpts']

print(f"Model rows: {len(pl_odds)}  (dropped {len(pl_model) - len(pl_odds)} with no odds match)")
print(f"\nSeason coverage:")
print(pl_odds.groupby(pl_odds['date'].dt.year.astype(str))['pts_vs_xpts'].count())

print(f"\npts_vs_xpts stats:")
print(pl_odds['pts_vs_xpts'].describe().round(3).to_string())

print(f"\nMean pts_vs_xpts by free_midweek:")
print(pl_odds.groupby('free_midweek')['pts_vs_xpts'].agg(['mean', 'count']).round(3).to_string())

import statsmodels.formula.api as smf

m = smf.ols(
    'pts_vs_xpts ~ free_midweek + rest_advantage + is_home',
    data=pl_odds
).fit()

print("\n=== OLS: pts_vs_xpts ~ free_midweek + rest_advantage + is_home ===")
for v in ['Intercept', 'free_midweek', 'rest_advantage', 'is_home']:
    if v in m.params.index:
        p   = m.pvalues[v]
        sig = '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.1 else ''))
        print(f"  {v:<30} coef={m.params[v]:+.3f}  p={p:.3f}  {sig}")
print(f"  R2 = {m.rsquared:.3f}  N = {int(m.nobs)}")

# midweek_adv variant (relative: +1 if only we had free midweek, -1 if only opponent)
m2 = smf.ols(
    'pts_vs_xpts ~ midweek_adv + rest_advantage + is_home',
    data=pl_odds
).fit()

print("\n=== OLS: midweek_adv variant ===")
for v in ['Intercept', 'midweek_adv', 'rest_advantage', 'is_home']:
    if v in m2.params.index:
        p   = m2.pvalues[v]
        sig = '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.1 else ''))
        print(f"  {v:<30} coef={m2.params[v]:+.3f}  p={p:.3f}  {sig}")
print(f"  R2 = {m2.rsquared:.3f}  N = {int(m2.nobs)}")


Model rows: 4022  (dropped 106 with no odds match)

Season coverage:
date
2017    224
2018    480
2019    522
2020    398
2021    522
2022    482
2023    574
2024    528
2025    292
Name: pts_vs_xpts, dtype: int64

pts_vs_xpts stats:
count    4022.000
mean        0.004
std         1.194
min        -2.665
25%        -0.944
50%        -0.282
75%         1.060
max         2.763

Mean pts_vs_xpts by free_midweek:
               mean  count
free_midweek              
0            -0.009   1659
1             0.013   2363

=== OLS: pts_vs_xpts ~ free_midweek + rest_advantage + is_home ===
  Intercept                      coef=+0.009  p=0.807  
  free_midweek                   coef=+0.005  p=0.904  
  rest_advantage                 coef=+0.007  p=0.471  
  is_home                        coef=-0.017  p=0.647  
  R2 = 0.000  N = 4022

=== OLS: midweek_adv variant ===
  Intercept                      coef=+0.013  p=0.637  
  midweek_adv                    coef=-0.020  p=0.777  
  rest_advantage  